- https://verl.readthedocs.io/en/latest/examples/config.html

### actor

#### use_dynamic_bsz

```python
# dp_actor.py
for _ in range(self.config.ppo_epochs):
    for batch_idx, mini_batch in enumerate(mini_batches):
        if self.config.use_dynamic_bsz:
            max_token_len = self.config.ppo_max_token_len_per_gpu * self.ulysses_sequence_parallel_size
            micro_batches, _ = prepare_dynamic_batch(mini_batch, max_token_len=max_token_len)
        else:
            self.gradient_accumulation = (
                self.config.ppo_mini_batch_size // self.config.ppo_micro_batch_size_per_gpu
            )
            micro_batches = mini_batch.split(self.config.ppo_micro_batch_size_per_gpu)

        self.actor_optimizer.zero_grad()
```


- 在 PPO 训练的 Actor 更新阶段，如何将一个较大的 mini_batch 切分为多个 micro_batch 以进行梯度累积（Gradient Accumulation）。
- 当 use_dynamic_bsz = False (Static 模式）：严格按照序列个数切分。
    - 梯度累积步数：固定为 `mini_batch_size // micro_batch_size`。
    - 如果数据中长短序列差异很大，长序列的 batch 可能导致 OOM（显存溢出），而短序列的 batch 则显存空闲，利用率低。
- use_dynamic_bsz = True：按照Token 总量切分。
    - 调用 `prepare_dynamic_batch`，贪心地将样本填入 micro-batch，直到该 batch 的总 Token 数达到上限（max_token_len）。

假设当前 Mini-batch 为集合 $\mathcal{B}$，包含 $N$ 个序列样本 $\{s_1, s_2, ..., s_N\}$。定义函数 $L(s_i)$ 为序列 $s_i$ 的 Token 长度。
我们需要将 $\mathcal{B}$ 划分为 $M$ 个互斥的 Micro-batch $\{b_1, b_2, ..., b_M\}$，使得 $\bigcup_{j=1}^M b_j = \mathcal{B}$。
- Case 1: Static Batching (use_dynamic_bsz = False)
    - 约束条件是每个 micro-batch 的序列数量固定为常数 $K$（即 ppo_micro_batch_size_per_gpu）：
    $$
    |b_j| = K=\frac{N}{M}, \quad \forall j \in \{1, ..., M\}
    $$
    - 此时梯度累积的 Loss 缩放系数为常数：
    $$
    \text{Scale}_j = \frac{1}{M}
    $$
- Case 2: Dynamic Batching (use_dynamic_bsz = True)
    - 约束条件是每个 micro-batch 的Token 总量不超过阈值 $T_{max}$（即 max_token_len）：
    $$
    \sum_{s \in b_j} L(s) \le T_{max}, \quad \forall j
    $$
    - 在此约束下，每个 micro-batch 包含的序列数量 $|b_j|$ 是动态变化的。为了保证对 Mini-batch $\mathcal{B}$ 的梯度估计是无偏的（即等价于在整个 $\mathcal{B}$ 上求平均），第 $j$ 个 micro-batch 的 Loss 缩放系数必须动态调整为：
    $$
    \text{Scale}j = \frac{|b_j|}{N} = \frac{\text{Count}(b_j)}{\text{Count}(\mathcal{B})}
    $$
    这对应了代码中的：`loss_scale_factor = response_mask.shape[0] / self.config.ppo_mini_batch_size`

- ga scale factor 的讨论：因为每个 Micro-batch 已经除以了 $K$（求了局部平均），为了得到全局平均（除以 $N$），我们需要再除以 $M$（因为 $1/K \times 1/M = 1/N$）。
    $$ \mathcal{L}_{total} = \frac{1}{N} \sum_{i=1}^{N} \ell_i $$
    - 在训练中，我们分 $M$ 次计算。第 $j$ 次 Micro-batch 计算出的 Loss（PyTorch 默认是 Mean Reduction）是该 Micro-batch 内 $K$ 个样本的平均值：
    $$ \mathcal{L}_{micro\_j} = \frac{1}{K}\sum_{x \in b_j} \ell(x) $$
    - 我们通过梯度累积将这 $M$ 次的结果加起来，为了还原出 $\mathcal{L}_{total}$，我们需要给每次累积乘以系数 $\text{scale}$：

$$
\begin{aligned}
\text{Accumulated Loss} &= \sum_{j=1}^{M} (\text{scale} \times \mathcal{L}_{micro\_j}) \\
&= \text{scale} \times \sum_{j=1}^{M} \left( \frac{1}{K} \sum_{x \in b_j} \ell(x) \right) \\
&= \text{scale} \times \frac{1}{K} \underbrace{\sum_{j=1}^{M} \sum_{x \in b_j} \ell(x)}{\text{所有 N 个样本的 Loss 之和}} \\
&= \text{scale} \times \frac{1}{K} \times (N \times \mathcal{L}_{total})
\end{aligned}
$$

- 为了让 $\text{Accumulated Loss} = \mathcal{L}_{total}$，我们需要：
$$ \text{scale} \times \frac{N}{K} = 1 \implies \text{scale} = \frac{K}{N} $$
又因为 $N = M \times K$，所以：
$$ \text{scale} = \frac{K}{M \times K} = \frac{1}{M} $$